# Train model dịch Trung→Việt chạy trên điện thoại

Notebook fine-tune **MoxhiMT-60** (Marian 57M, Apache-2.0) với dữ liệu của bạn.
Chạy trên **GPU free của Colab**: Runtime → Change runtime type → T4 GPU.

Chuẩn bị dữ liệu trước: `data/series/<ten-bo>/<ten-bo>.zh.txt` + `.vi.txt` (câu lệch dòng)
hoặc `data/train.jsonl` (mỗi dòng `{"zh": ..., "vi": ...}`). Đẩy dữ liệu lên Drive rồi mount.

In [ ]:
!pip -q install transformers sentencepiece datasets sacrebleu ctranslate2 optimum onnxruntime
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG — bật GPU trong Runtime menu!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# giả sử project nằm ở: /content/drive/MyDrive/D-ch-local-hachimi
import os
ROOT = '/content/drive/MyDrive/D-ch-local-hachimi'
os.chdir(ROOT); !ls

In [ ]:
# 1) Gộp dữ liệu từng bộ thành train/val.jsonl
!python scripts/make_series_data.py --merge-all

In [ ]:
# 2) Fine-tune từ MoxhiMT-60 (bộ nhỏ vài nghìn câu: vài phút; vài trăm nghìn câu: vài giờ)
!python scripts/train.py --mode finetune --init DanVP/MoxhiMT-60 \
  --train data/series/all/train.jsonl --val data/series/all/val.jsonl \
  --out out/model --epochs 2 --batch 32

In [ ]:
# 3) Đánh giá so với model gốc
!python scripts/evaluate.py --model out/model --base DanVP/MoxhiMT-60 --val data/series/all/val.jsonl --n 200

In [ ]:
# 4) Xuất bản INT8 cho box (CTranslate2) và bản ONNX cho điện thoại
!python scripts/export_ct2.py --model out/model --out out/model-ct2-int8
!python scripts/export_onnx.py --model out/model --out out/model-onnx
!du -sh out/*

In [ ]:
# 5) Nén lại để tải về / đẩy lên Drive — gửi lại cho tôi deploy lên box
!cd out && zip -q -r model.zip model model-ct2-int8 model-onnx
!cp out/../out/model.zip /content/drive/MyDrive/ 2>/dev/null || true
!du -sh out/model.zip